# 02 — Normalise the extract into keyed tables and load them into SQLite

**Reads:** `data/processed/psid_wide.parquet`, `data/raw/psid_variable_crosswalk.csv`, `data/raw/cpi_u_annual.csv`.
**Writes:** `data/processed/psid.sqlite` with three tables, plus `results/tables/table_counts.csv`.

The Data Center pastes each year's family record onto every member's row, so one wide row per person holds
43 waves of both person-level and family-level columns. This notebook undoes that into the tables PSID
conceptually has:

| table | one row per | columns |
|---|---|---|
| `individuals` | person | `id_1968, person_num, sex, father_id_1968, father_person_num, mother_id_1968, mother_person_num` |
| `persons` | person × year | `id_1968, person_num, year, interview, age, rel_head, seq` |
| `families` | family × year | `year, fam_interview, fam_income, income_year, fam_income_real` |

Each step ends with a count — the Class 12 rule: validate row counts and key uniqueness after every reshape or join.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent
RAW, PROC, TAB = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "results" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

wide = pd.read_parquet(PROC / "psid_wide.parquet")
cw = pd.read_csv(RAW / "psid_variable_crosswalk.csv")
cpi = pd.read_csv(RAW / "cpi_u_annual.csv")
wide.shape

## 1. `individuals` — the fixed columns
Rows = people. Key = (`id_1968`, `person_num`). Check the key is unique before anything else.

## 2. `persons` — wide → long on the four per-year person variables
Target: one row per person × year with `interview, age, rel_head, seq`. The crosswalk gives you, for each
column name, its `concept` and `year`; stack the 43 wave-blocks. Keep only rows where the person was interviewed
(`interview > 0`). Then check: how many rows, and is (`id_1968`, `person_num`, `year`) unique?

## 3. `families` — wide → long on the two per-year family variables, then dedupe
Target: one row per family × year. After melting, the same family-year appears once per member; drop the repeats.
Then the uniqueness check on (`year`, `fam_interview`): it will **not** pass cleanly in every year.
Find the offending rows (which `seq` values do those people have?) and decide — and write down — how to treat them.
Add `income_year = year - 1` and `fam_income_real` using the CPI deflator for the income year.

## 4. Load into SQLite and record the counts
`sqlite3.connect(PROC / "psid.sqlite")`, `DataFrame.to_sql(..., if_exists="replace", index=False)`.
Save one small table of row counts per SQL table to `results/tables/table_counts.csv` — it goes in the post.